#환경설정

In [1]:
!pip install langchain-openai langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 22.3 MB/s eta 0:00:00
  Attempting uninstall: langchain-protocol
    Found existing installation: langchain-protocol 0.0.16
    Uninstalling langchain-protocol-0.0.16:
      Successfully uninstalled langchain-protocol-0.0.16
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.3
    Uninstalling langchain-core-1.4.3:
      Successfully uninstalled langchain-core-1.4.3


In [10]:
!pip install sqlglot -q

In [11]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from validate_dataset import validate_json_file
import json
import re
import pandas as pd
import random
import sqlite3
import os

In [ ]:
os.environ["OPENAI_API_KEY"] = ""
llm_sql = ChatOpenAI(model = 'gpt-4o-mini', temperature=0)

In [9]:
def _convert_to_sqlite(sql: str) -> str:
    """
    DB 종류에 따라 다른 SQL문을 LLM을 사용해 SQLite 문법으로 변환.
    execute_sql_on_db 에서만 사용하는 함수

    Args:
        sql : 변환할 SQL문

    Returns:
        SQLite 문법으로 변환된 SQL문
    """
    response = llm_sql.invoke(
        f"""다음 SQL문을 SQLite 문법으로 변환해줘.
반드시 SQL문만 출력하고 다른 설명은 절대 추가하지 마.
백틱이나 코드블록 없이 순수 SQL문만 출력해.

{sql}"""
    )
    return response.content.strip()

#데이터 평가/검증



```
# 데이터 형식
1. 모든 list의 원소가 dictionary type인지
2. instruction, input, output 키 존재여부
3. instruction과 output의 값 null 여부

# instruction 값
1. 값이 string 형인지 확인
2. 값에 '입력 텍스트', 'DDL statements' 존재여부
3. '입력 텍스트'가 항상 'DDL statements'보다 앞에 오는지

DDL statements에 INSERT문이 있다면
1. CREATE문에 적힌 테이블명과 INSERT 문에 쓰인 테이블명이 일치하는지
2. CREATE문에 적힌 칼럼명과 INSERT 문에 쓰인 칼럼명이 전체 일치하는지
3. INSERT문에 쓰인 칼럼 개수와 값의 개수가 일치하는지.
4. VALUES 값이 칼럼 데이터형에 맞는 올바른 자료형인지
5. VALUES 값이 칼럼 NULL 허용 여부에 맞는지.
6. 전체 INSERT을 봤을 때 PK의 중복 여부

#input 값
1. 항상 빈 문자열인지 체크

# output 값
1. 값이 string 형인지 확인
2. '쿼리 작성' 존재여부
SQL 확인
1. SQL 실제 실행 되는지 여부
2. SQL이 참조하는 컬럼이 DDL statements에 정의된 칼럼인지 여부

#중복 여부
1. 전체 데이터에서 instruction이 중복되는 것이 있는지 확인
2. 전체 데이터에서 output의 쿼리문이 중복되는 것이 있는지 확인

XXXXXXXXXXXXXXXXXXXX
삭제
전체 데이터에서 instruction의 DDL statement가 중복되는 것이 있는지 확인
XXXXXXXXXXXXXXXXXXXX
```

# 단일 DB SQL 데이터

##1. 전체 통과

In [ ]:
report_orders = validate_json_file("data/Olist_orders_text_to_sql_data.json", llm_sql=llm_sql, _convert_to_sqlite=_convert_to_sqlite)
report_order_items = validate_json_file("data/Olist_order_items_text_to_sql_data.json",llm_sql=llm_sql,_convert_to_sqlite=_convert_to_sqlite)
report_order_payments = validate_json_file("data/Olist_order_payments_text_to_sql_data.json",llm_sql=llm_sql,_convert_to_sqlite=_convert_to_sqlite)
report_products = validate_json_file("data/Olist_products_text_to_sql_data.json",llm_sql=llm_sql,_convert_to_sqlite=_convert_to_sqlite)
report_sellers = validate_json_file("data/Olist_sellers_text_to_sql_data.json",llm_sql=llm_sql,_convert_to_sqlite=_convert_to_sqlite)
report_customers = validate_json_file("data/Olist_customers_text_to_sql_data.json",llm_sql=llm_sql,_convert_to_sqlite=_convert_to_sqlite)

파일: data/Olist_orders_text_to_sql_data.json
총 100건 로드

[데이터 형식 - 원소가 dict 타입인지] 전체 통과
[데이터 형식 - instruction/input/output 키 존재] 전체 통과
[데이터 형식 - instruction 값 not null] 전체 통과
[데이터 형식 - output 값 not null] 전체 통과
[instruction 값 - 값이 string 타입인지] 전체 통과
[instruction 값 - '입력 텍스트' 존재] 전체 통과
[instruction 값 - 'DDL statements' 존재] 전체 통과
[instruction 값 - '입력 텍스트'가 'DDL statements'보다 앞에 위치] 전체 통과
[instruction(INSERT) - CREATE/INSERT 테이블명 일치] 전체 통과
[instruction(INSERT) - CREATE/INSERT 컬럼명 전체 일치] 전체 통과
[instruction(INSERT) - 컬럼 개수와 값 개수 일치] 전체 통과
[instruction(INSERT) - VALUES 값이 컬럼 데이터형에 맞는지] 전체 통과
[instruction(INSERT) - VALUES 값이 NULL 허용 여부에 맞는지] 전체 통과
[instruction(INSERT) - PRIMARY KEY 값 중복 여부] 전체 통과
[input 값 - 항상 빈 문자열인지] 전체 통과
[output 값 - 값이 string 타입인지] 전체 통과
[output 값 - '쿼리 작성' 존재] 전체 통과
[output(SQL) - SQL이 참조하는 컬럼이 DDL에 정의되어 있는지] 전체 통과
[output(SQL) - SQL 실제 실행 가능 여부] 전체 통과
[중복 여부 - instruction 중복] 전체 통과
[중복 여부 - output SQL 중복] 전체 통과


##2. 오류 발생 - order_reviews

In [13]:
report_order_reviews = validate_json_file("data/Olist_order_reviews_text_to_sql_data.json",llm_sql=llm_sql,_convert_to_sqlite=_convert_to_sqlite)

파일: data/Olist_order_reviews_text_to_sql_data.json
총 100건 로드

[데이터 형식 - 원소가 dict 타입인지] 전체 통과
[데이터 형식 - instruction/input/output 키 존재] 전체 통과
[데이터 형식 - instruction 값 not null] 전체 통과
[데이터 형식 - output 값 not null] 전체 통과
[instruction 값 - 값이 string 타입인지] 전체 통과
[instruction 값 - '입력 텍스트' 존재] 전체 통과
[instruction 값 - 'DDL statements' 존재] 전체 통과
[instruction 값 - '입력 텍스트'가 'DDL statements'보다 앞에 위치] 전체 통과
[instruction(INSERT) - CREATE/INSERT 테이블명 일치] 전체 통과
[instruction(INSERT) - CREATE/INSERT 컬럼명 전체 일치] 전체 통과
[instruction(INSERT) - 컬럼 개수와 값 개수 일치] 전체 통과
[instruction(INSERT) - VALUES 값이 컬럼 데이터형에 맞는지] 전체 통과
[instruction(INSERT) - VALUES 값이 NULL 허용 여부에 맞는지] 전체 통과
[instruction(INSERT) - PRIMARY KEY 값 중복 여부] 전체 통과
[input 값 - 항상 빈 문자열인지] 전체 통과
[output 값 - 값이 string 타입인지] 전체 통과
[output 값 - '쿼리 작성' 존재] 전체 통과
[output(SQL) - SQL이 참조하는 컬럼이 DDL에 정의되어 있는지] 전체 통과
[output(SQL) - SQL 실제 실행 가능 여부] 위반 2건
  - index=75 | 1차 오류=no such function: SUBSTRING_INDEX / LLM 변환 후 오류=near "ORDER": syntax error
  - index=79 | 1차 오류

In [14]:
# 1. JSON 파일 읽기
with open("data/Olist_order_reviews_text_to_sql_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [19]:
# 2. 오류 내용 실제 출력해보기.
original = data[75]['output']
converted = _convert_to_sqlite(data[75]['output'])

print(original)
print('--------------------')
print(converted)

쿼리 작성: SELECT COUNT(*) AS order_count
FROM (
  SELECT
    order_id,
    SUBSTRING_INDEX(GROUP_CONCAT(review_score ORDER BY review_answer_timestamp ASC), ',', 1) AS first_score,
    SUBSTRING_INDEX(GROUP_CONCAT(review_score ORDER BY review_answer_timestamp DESC), ',', 1) AS last_score
  FROM order_reviews
  GROUP BY order_id
) x
WHERE first_score <> last_score;
SELECT COUNT(*) AS order_count
FROM (
  SELECT
    order_id,
    SUBSTR(GROUP_CONCAT(review_score ORDER BY review_answer_timestamp ASC), 1, INSTR(GROUP_CONCAT(review_score ORDER BY review_answer_timestamp ASC), ',') - 1) AS first_score,
    SUBSTR(GROUP_CONCAT(review_score ORDER BY review_answer_timestamp DESC), 1, INSTR(GROUP_CONCAT(review_score ORDER BY review_answer_timestamp DESC), ',') - 1) AS last_score
  FROM order_reviews
  GROUP BY order_id
) x
WHERE first_score <> last_score;


In [21]:
# 2. 오류 내용 실제 출력해보기.
original = data[79]['output']
converted = _convert_to_sqlite(data[79]['output'])

print(original)
print('--------------------')
print(converted)

쿼리 작성: SELECT
  DATE_FORMAT(review_creation_date, '%Y-%m') AS month,
  AVG(CHAR_LENGTH(review_comment_title)) AS avg_title_length,
  AVG(CHAR_LENGTH(review_comment_message)) AS avg_message_length
FROM order_reviews
WHERE review_creation_date >= '2018-01-01' AND review_creation_date < '2019-01-01'
  AND review_comment_title IS NOT NULL AND TRIM(review_comment_title) <> ''
  AND review_comment_message IS NOT NULL AND TRIM(review_comment_message) <> ''
GROUP BY DATE_FORMAT(review_creation_date, '%Y-%m')
ORDER BY month;
--------------------
SELECT
  strftime('%Y-%m', review_creation_date) AS month,
  AVG LENGTH(review_comment_title) AS avg_title_length,
  AVG LENGTH(review_comment_message) AS avg_message_length
FROM order_reviews
WHERE review_creation_date >= '2018-01-01' AND review_creation_date < '2019-01-01'
  AND review_comment_title IS NOT NULL AND TRIM(review_comment_title) <> ''
  AND review_comment_message IS NOT NULL AND TRIM(review_comment_message) <> ''
GROUP BY strftime('%Y-%m'

In [22]:
#3. index 75, 79 삭제

indices_to_remove = sorted([75, 79], reverse=True)
for idx in indices_to_remove:
    removed = data.pop(idx)
    print(f"index {idx} 삭제 완료")

print(f"삭제 후 데이터 개수: {len(data)}")

# 3. 같은 형식(JSON, UTF-8, 한글 보존, 들여쓰기 2칸)으로 저장
with open("Olist_order_reviews_text_to_sql_data.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)


index 79 삭제 완료
index 75 삭제 완료
삭제 후 데이터 개수: 98


In [23]:
#재확인
report_order_reviews = validate_json_file("Olist_order_reviews_text_to_sql_data.json",llm_sql=llm_sql,_convert_to_sqlite=_convert_to_sqlite)

파일: Olist_order_reviews_text_to_sql_data.json
총 98건 로드

[데이터 형식 - 원소가 dict 타입인지] 전체 통과
[데이터 형식 - instruction/input/output 키 존재] 전체 통과
[데이터 형식 - instruction 값 not null] 전체 통과
[데이터 형식 - output 값 not null] 전체 통과
[instruction 값 - 값이 string 타입인지] 전체 통과
[instruction 값 - '입력 텍스트' 존재] 전체 통과
[instruction 값 - 'DDL statements' 존재] 전체 통과
[instruction 값 - '입력 텍스트'가 'DDL statements'보다 앞에 위치] 전체 통과
[instruction(INSERT) - CREATE/INSERT 테이블명 일치] 전체 통과
[instruction(INSERT) - CREATE/INSERT 컬럼명 전체 일치] 전체 통과
[instruction(INSERT) - 컬럼 개수와 값 개수 일치] 전체 통과
[instruction(INSERT) - VALUES 값이 컬럼 데이터형에 맞는지] 전체 통과
[instruction(INSERT) - VALUES 값이 NULL 허용 여부에 맞는지] 전체 통과
[instruction(INSERT) - PRIMARY KEY 값 중복 여부] 전체 통과
[input 값 - 항상 빈 문자열인지] 전체 통과
[output 값 - 값이 string 타입인지] 전체 통과
[output 값 - '쿼리 작성' 존재] 전체 통과
[output(SQL) - SQL이 참조하는 컬럼이 DDL에 정의되어 있는지] 전체 통과
[output(SQL) - SQL 실제 실행 가능 여부] 전체 통과
[중복 여부 - instruction 중복] 전체 통과
[중복 여부 - output SQL 중복] 전체 통과


##3. 오류 발생 - geolocation

In [29]:
report_geolocation = validate_json_file("data/Olist_geolocation_text_to_sql_data.json",llm_sql=llm_sql,_convert_to_sqlite=_convert_to_sqlite)

파일: data/Olist_geolocation_text_to_sql_data.json
총 100건 로드

[데이터 형식 - 원소가 dict 타입인지] 전체 통과
[데이터 형식 - instruction/input/output 키 존재] 전체 통과
[데이터 형식 - instruction 값 not null] 전체 통과
[데이터 형식 - output 값 not null] 전체 통과
[instruction 값 - 값이 string 타입인지] 전체 통과
[instruction 값 - '입력 텍스트' 존재] 전체 통과
[instruction 값 - 'DDL statements' 존재] 전체 통과
[instruction 값 - '입력 텍스트'가 'DDL statements'보다 앞에 위치] 전체 통과
[instruction(INSERT) - CREATE/INSERT 테이블명 일치] 전체 통과
[instruction(INSERT) - CREATE/INSERT 컬럼명 전체 일치] 전체 통과
[instruction(INSERT) - 컬럼 개수와 값 개수 일치] 전체 통과
[instruction(INSERT) - VALUES 값이 컬럼 데이터형에 맞는지] 전체 통과
[instruction(INSERT) - VALUES 값이 NULL 허용 여부에 맞는지] 전체 통과
[input 값 - 항상 빈 문자열인지] 전체 통과
[output 값 - 값이 string 타입인지] 전체 통과
[output 값 - '쿼리 작성' 존재] 전체 통과
[output(SQL) - SQL이 참조하는 컬럼이 DDL에 정의되어 있는지] 전체 통과
[output(SQL) - SQL 실제 실행 가능 여부] 위반 3건
  - index=57 | 1차 오류=no such function: STDDEV_POP / LLM 변환 후 오류=no such column: avg_table.geolocation_state
  - index=83 | 1차 오류=no such function: STDDEV_SAMP / LLM 변환 

In [35]:
# 1. JSON 파일 읽기
with open("data/Olist_geolocation_text_to_sql_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [31]:
# 2. 오류 내용 실제 출력해보기.
original = data[57]['output']
converted = _convert_to_sqlite(data[57]['output'])

print(original)
print('--------------------')
print(converted)

쿼리 작성: SELECT
  geolocation_state,
  STDDEV_POP(geolocation_lat) AS lat_stddev
FROM geolocation
GROUP BY geolocation_state
ORDER BY lat_stddev DESC, geolocation_state ASC;
--------------------
SELECT
  geolocation_state,
  STDEV(geolocation_lat) AS lat_stddev
FROM geolocation
GROUP BY geolocation_state
ORDER BY lat_stddev DESC, geolocation_state ASC;


In [32]:
# 2. 오류 내용 실제 출력해보기.
original = data[83]['output']
converted = _convert_to_sqlite(data[83]['output'])

print(original)
print('--------------------')
print(converted)

쿼리 작성: SELECT
  geolocation_state,
  AVG(geolocation_lng) AS avg_lng,
  STDDEV_SAMP(geolocation_lng) AS stddev_lng
FROM geolocation
GROUP BY geolocation_state
ORDER BY stddev_lng DESC, geolocation_state;
--------------------
SELECT
  geolocation_state,
  AVG(geolocation_lng) AS avg_lng,
  (SELECT sqrt(sum((geolocation_lng - avg_lng) * (geolocation_lng - avg_lng)) / (count(*) - 1)) FROM geolocation g2 WHERE g2.geolocation_state = g1.geolocation_state) AS stddev_lng
FROM geolocation g1
GROUP BY geolocation_state
ORDER BY stddev_lng DESC, geolocation_state;


In [34]:
# 2. 오류 내용 실제 출력해보기.
original = data[96]['output']
converted = _convert_to_sqlite(data[96]['output'])

print(original)
print('--------------------')
print(converted)

쿼리 작성: WITH city_counts AS (
  SELECT
    geolocation_state,
    geolocation_city,
    COUNT(*) AS zip_count
  FROM geolocation
  GROUP BY geolocation_state, geolocation_city
)
SELECT
  geolocation_state,
  STDDEV_SAMP(zip_count) AS zip_count_stddev
FROM city_counts
GROUP BY geolocation_state
ORDER BY zip_count_stddev DESC, geolocation_state
LIMIT 5;
--------------------
WITH city_counts AS (
  SELECT
    geolocation_state,
    geolocation_city,
    COUNT(*) AS zip_count
  FROM geolocation
  GROUP BY geolocation_state, geolocation_city
)
SELECT
  geolocation_state,
  (SELECT sqrt(sum((zip_count - avg(zip_count)) * (zip_count - avg(zip_count))) / (count(*) - 1)) FROM city_counts) AS zip_count_stddev
FROM city_counts
GROUP BY geolocation_state
ORDER BY zip_count_stddev DESC, geolocation_state
LIMIT 5;


In [36]:
#3. index 57, 83, 96 삭제

indices_to_remove = sorted([57, 83, 96], reverse=True)
for idx in indices_to_remove:
    removed = data.pop(idx)
    print(f"index {idx} 삭제 완료")

print(f"삭제 후 데이터 개수: {len(data)}")

# 3. 같은 형식(JSON, UTF-8, 한글 보존, 들여쓰기 2칸)으로 저장
with open("Olist_geolocation_text_to_sql_data.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)


index 96 삭제 완료
index 83 삭제 완료
index 57 삭제 완료
삭제 후 데이터 개수: 97


In [37]:
#재확인
report_order_reviews = validate_json_file("Olist_geolocation_text_to_sql_data.json",llm_sql=llm_sql,_convert_to_sqlite=_convert_to_sqlite)

파일: Olist_geolocation_text_to_sql_data.json
총 97건 로드

[데이터 형식 - 원소가 dict 타입인지] 전체 통과
[데이터 형식 - instruction/input/output 키 존재] 전체 통과
[데이터 형식 - instruction 값 not null] 전체 통과
[데이터 형식 - output 값 not null] 전체 통과
[instruction 값 - 값이 string 타입인지] 전체 통과
[instruction 값 - '입력 텍스트' 존재] 전체 통과
[instruction 값 - 'DDL statements' 존재] 전체 통과
[instruction 값 - '입력 텍스트'가 'DDL statements'보다 앞에 위치] 전체 통과
[instruction(INSERT) - CREATE/INSERT 테이블명 일치] 전체 통과
[instruction(INSERT) - CREATE/INSERT 컬럼명 전체 일치] 전체 통과
[instruction(INSERT) - 컬럼 개수와 값 개수 일치] 전체 통과
[instruction(INSERT) - VALUES 값이 컬럼 데이터형에 맞는지] 전체 통과
[instruction(INSERT) - VALUES 값이 NULL 허용 여부에 맞는지] 전체 통과
[input 값 - 항상 빈 문자열인지] 전체 통과
[output 값 - 값이 string 타입인지] 전체 통과
[output 값 - '쿼리 작성' 존재] 전체 통과
[output(SQL) - SQL이 참조하는 컬럼이 DDL에 정의되어 있는지] 전체 통과
[output(SQL) - SQL 실제 실행 가능 여부] 전체 통과
[중복 여부 - instruction 중복] 전체 통과
[중복 여부 - output SQL 중복] 전체 통과
